In [1]:
"""
05a_gr4j_preprocessing.ipynb
------------------------------
Prepares daily time series inputs for GR4J benchmark per station:
    P    = rain + snowfall (rainUps + snowUps) [mm/day]
    ET0  = reference evapotranspiration (etUps) [mm/day]
    Q_mm = observed discharge converted to mm/day

Output: one parquet per station → DIR_OUT / gr4j_inputs / {ID}.parquet
"""

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [2]:
# =============================================================================
# HELPER Function(s)
# =============================================================================

def read_tss_single_gauge(tss_path: Path, start_date: str) -> pd.Series:
    if not tss_path.exists():
        return pd.Series(dtype=float)
    with open(tss_path) as f:
        lines = f.readlines()
    try:
        n_scalars = int(lines[1].strip())
    except (IndexError, ValueError):
        n_scalars = 1
    skiprows = 2 + n_scalars
    try:
        df = pd.read_csv(
            tss_path, skiprows=skiprows, sep=r'\s+',
            header=None, usecols=[0, 1],
            dtype={0: float, 1: float},
        )
        dates  = pd.date_range(start=start_date, periods=len(df), freq="D")
        series = pd.Series(df.iloc[:, 1].values, index=dates, dtype=float)
        series = series.replace(-9999.0, np.nan)
        series = series.where(series > -9000)
        return series
    except Exception as e:
        print(f"    ⚠️  Could not parse {tss_path.name}: {e}")
        return pd.Series(dtype=float)

## Config

In [3]:
# =============================================================================
# CONFIG
# =============================================================================

BASE_FILE = Path.cwd() / "../glofas5_hydrobot.csv"
DIR_TSS    = Path("/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/long_term_runs/01_Hydrology/")
DIR_OUT    = Path.cwd() / "../gr4j/inputs"
DIR_OUT.mkdir(parents=True, exist_ok=True)

RUN_NAME   = "long_term_run"
START_DATE = "1975-01-02"
CAL_START  = "1991-01-01"
CAL_END    = "2020-12-31"

# Variables needed for GR4J
VARS_GR4J = {
    "rain": "rainUps",
    "sf":   "snowUps",
    "et0":  "etUps",
    "q":    "dis",
}

In [4]:
# =============================================================================
# LOAD BASE INFO
# =============================================================================

print("Loading base station info...")
glofas5_base_info = pd.read_csv(BASE_FILE)
glofas5_base_info["ID"] = glofas5_base_info["ID"].astype(int)
station_ids = set(glofas5_base_info["ID"].tolist())

# Area lookup for Q conversion
area_lookup = glofas5_base_info.set_index("ID")["DrainageArea_LDD"]

print(f"  {len(station_ids)} stations in base file")

Loading base station info...
  5379 stations in base file


## Load TSS

In [5]:
# =============================================================================
# DISCOVER TSS FOLDERS
# =============================================================================

print(f"\nScanning TSS directory: {DIR_TSS}")
station_folders = {}
for tss_file in DIR_TSS.rglob(f"dis{RUN_NAME}.tss"):
    folder = tss_file.parent
    try:
        sid = int(folder.parts[-3])
        if sid in station_ids:
            station_folders[sid] = folder
    except (ValueError, IndexError):
        continue

print(f"  Found TSS folders for {len(station_folders)} / {len(station_ids)} stations")
missing_folders = station_ids - set(station_folders.keys())
if missing_folders:
    pd.DataFrame(sorted(missing_folders), columns=["ID"]).to_csv(
        Path.cwd() / "gr4j_missing_tss.csv", index=False)
    print(f"  ⚠️  Missing {len(missing_folders)} stations → saved to gr4j_missing_tss.csv")


Scanning TSS directory: /mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/long_term_runs/01_Hydrology
  Found TSS folders for 5262 / 5379 stations
  ⚠️  Missing 117 stations → saved to gr4j_missing_tss.csv


In [6]:

# =============================================================================
# MAIN LOOP — one parquet per station
# =============================================================================

skipped   = []
processed = 0

for sid in tqdm(sorted(station_folders.keys()), desc="Preprocessing GR4J inputs"):
    folder = station_folders[sid]

    # ── Read all 4 variables ──
    ts = {}
    for varname, tss_prefix in VARS_GR4J.items():
        tss_path = folder / f"{tss_prefix}{RUN_NAME}.tss"
        ts[varname] = read_tss_single_gauge(tss_path, START_DATE)

    # ── Check all variables are available ──
    if any(s.empty for s in ts.values()):
        missing_vars = [v for v, s in ts.items() if s.empty]
        skipped.append({"ID": sid, "reason": f"missing: {missing_vars}"})
        continue

    # ── Build DataFrame ──
    df_out = pd.DataFrame({
        "rain": ts["rain"],
        "sf":   ts["sf"],
        "et0":  ts["et0"],
        "q":    ts["q"],
    })

    # ── P = rain + snowfall [mm/day] ──
    df_out["P"] = df_out["rain"] + df_out["sf"]

    # ── Q_mm = m³/s → mm/day ──
    area_km2 = area_lookup.get(sid, np.nan)
    if np.isfinite(area_km2) and area_km2 > 0:
        df_out["Q_mm"] = df_out["q"] * 86400 * 1000 / (area_km2 * 1e6)
    else:
        df_out["Q_mm"] = np.nan

    # ── Clip to calibration period ──
    df_out = df_out.loc[CAL_START:CAL_END, ["P", "et0", "Q_mm"]]
    df_out.columns = ["P", "ET0", "Q_mm"]
    df_out.index.name = "date"

    # ── Sanity checks ──
    if df_out[["P", "ET0", "Q_mm"]].isna().all().any():
        skipped.append({"ID": sid, "reason": "all NaN in at least one variable"})
        continue

    # ── Save ──
    df_out.to_parquet(DIR_OUT / f"{sid}.parquet")
    processed += 1

Preprocessing GR4J inputs:   0%|          | 0/5262 [00:00<?, ?it/s]


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print(f"\nDone!")
print(f"  Processed : {processed} stations")
print(f"  Skipped   : {len(skipped)} stations")

if skipped:
    pd.DataFrame(skipped).to_csv(Path.cwd() / "gr4j_skipped_stations.csv", index=False)
    print(f"  Skipped stations → gr4j_skipped_stations.csv")

# Quick sanity check on one station
sample_id = sorted(station_folders.keys())[0]
sample_df  = pd.read_parquet(DIR_OUT / f"{sample_id}.parquet")
print(f"\nSample station {sample_id}:")
print(sample_df.describe().round(3))
print(f"Date range: {sample_df.index[0]} → {sample_df.index[-1]}")
print(f"N days: {len(sample_df)}")